# Guided-PIBT: real routing baseline ( S5) — Colab/Kaggle

Validates our betweenness-penalty **routing proxy** against the *real* routing method it
stands for: **Guided-PIBT** = the official impl. of Chen et al. 2023 (Traffic Flow
Optimisation, our `\cite{trafficflow}`). We build it twice --- traffic-flow guidance
**ON** vs **OFF** (plain PIBT) --- and run its bundled **Paris** lifelong benchmarks
across agent counts, to see whether real routing recovers throughput under
over-saturation or also saturates.

CPU-only. **Build risk:** Guided-PIBT needs **Boost >= 1.83** (newer than apt's 1.74), so
we get it from conda-forge. Gate = the build cell (3) producing two `lifelong` binaries.
If the build fights back the way LaCAM did, stop and we fall back to citing their
published peak-density result.

## 1. Platform + clone

In [ ]:
import os
KAGGLE = os.path.exists('/kaggle')
BASE = '/kaggle/working' if KAGGLE else '/content'
GP = f'{BASE}/Guided-PIBT'
!apt-get -qq update && apt-get -qq install -y cmake build-essential >/dev/null
if not os.path.exists(GP):
    !git clone --recursive -q https://github.com/nobodyczcz/Guided-PIBT.git {GP}
!cd {GP} && git submodule update --init --recursive
print('cloned:', os.path.isdir(f'{GP}/guided-pibt'))

## 2. Boost 1.83 (conda-forge)

In [ ]:
import subprocess
if KAGGLE:
    if not os.path.exists('/kaggle/working/miniforge'):
        !wget -qO /tmp/mf.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
        !bash /tmp/mf.sh -b -p /kaggle/working/miniforge >/dev/null
    PREFIX = '/kaggle/working/miniforge'; CONDA = f'{PREFIX}/bin/conda'
else:
    try:
        import condacolab; condacolab.check()
    except Exception:
        !pip install -q condacolab
        import condacolab; condacolab.install()   # restarts kernel; re-run this cell after
    PREFIX = '/usr/local'; CONDA = 'conda'
# Boost 1.83 + a MATCHING conda compiler (avoids Boost.Log ABI mismatch with system gcc)
!{CONDA} install -y -c conda-forge 'libboost-devel=1.83' 'libboost-headers=1.83' gcc_linux-64 gxx_linux-64 cmake make >/tmp/dep.log 2>&1; tail -2 /tmp/dep.log
GCC = f'{PREFIX}/bin/x86_64-conda-linux-gnu-gcc'
GXX = f'{PREFIX}/bin/x86_64-conda-linux-gnu-g++'
RUNENV = dict(os.environ, LD_LIBRARY_PATH=f'{PREFIX}/lib:' + os.environ.get('LD_LIBRARY_PATH',''))
print('boost:', os.path.exists(f'{PREFIX}/include/boost/version.hpp'),
      '| conda g++:', os.path.exists(GXX), '| PREFIX =', PREFIX)

## 3. Build twice: routing ON vs OFF  *(gate: two `lifelong` binaries)*

In [ ]:
# CONFIG-mode Boost is found via CMAKE_PREFIX_PATH=conda; build with conda's compiler.
common = (f'-DCMAKE_PREFIX_PATH={PREFIX} '
          f'-DCMAKE_C_COMPILER={GCC} -DCMAKE_CXX_COMPILER={GXX} '
          '-DGUIDANCE=OFF -DGUIDANCE_LNS=OFF -DINIT_PP=OFF -DRELAX=OFF '
          '-DOBJECTIVE=2 -DFOCAL_SEARCH=OFF -DCMAKE_BUILD_TYPE=RELEASE')
CMAKE = f'{PREFIX}/bin/cmake'
for name, flag in [('on','-DFLOW_GUIDANCE=ON'), ('off','-DFLOW_GUIDANCE=OFF')]:
    log = f'/tmp/build_{name}.log'
    !cd {GP} && {CMAKE} -B build-{name} ./guided-pibt {common} {flag} >{log} 2>&1 && {CMAKE} --build build-{name} -j2 >>{log} 2>&1; tail -5 {log}
    print(f'{name}: lifelong built =', os.path.exists(f'{GP}/build-{name}/lifelong'), '\n')
assert os.path.exists(f'{GP}/build-on/lifelong') and os.path.exists(f'{GP}/build-off/lifelong'), 'build failed — paste tail of /tmp/build_on.log'

## 4. Discover the output schema (one quick run)

In [ ]:
import glob, json, subprocess
benches = sorted(glob.glob(f'{GP}/guided-pibt/benchmark-lifelong/Paris_1_256_0_*.json'))
print('benchmarks (seed 0):', [os.path.basename(b) for b in benches])
b0 = benches[0]
subprocess.run([f'{GP}/build-on/lifelong','--inputFile',b0,'--planTimeLimit','10','--output','/tmp/o.json'],
               cwd=GP, env=RUNENV)
o = json.load(open('/tmp/o.json'))
print('OUTPUT KEYS:', list(o.keys()))
for k in o:
    if not isinstance(o[k], (list, dict)): print(f'  {k} = {o[k]}')
bj = json.load(open(b0)); print('BENCH KEYS:', [k for k in bj])

## 5. Sweep ON vs OFF across agent counts
Adjust `AGENTS`/`SEEDS` to the filenames printed above. Throughput = tasks finished /
timesteps; tweak `tput()` to the real keys from cell 4 if needed.

In [ ]:
import re, time
RESULTS = '/kaggle/working/mapf-deadlock-results' if KAGGLE else f'{BASE}/mapf-deadlock-results'
OUT = f'{RESULTS}/guided-pibt'; os.makedirs(OUT, exist_ok=True)
def tput(o):
    # best-effort: tasks finished / number of timesteps
    fin = next((o[k] for k in o if 'finish' in k.lower() or 'numtask' in k.lower()), None)
    steps = next((o[k] for k in o if k.lower() in ('makespan','numtimestep','numtimesteps','finalstep','steps')), None)
    return (fin/steps) if (fin and steps) else (fin, steps)
SEEDS = [0]
rows = {}
for b in benches:
    n = int(re.search(r'_(\d+)\.json$', b).group(1))
    for mode in ['on','off']:
        outj = f'{OUT}/{mode}_n{n}.json'
        if not os.path.exists(outj):
            print(f'=== {mode} n{n} ===', flush=True); t0=time.time()
            subprocess.run([f'{GP}/build-{mode}/lifelong','--inputFile',b,'--planTimeLimit','10','--output',outj], cwd=GP, env=RUNENV)
            print('   ', f'{(time.time()-t0)/60:.1f} min', flush=True)
        try: rows.setdefault(n, {})[mode] = tput(json.load(open(outj)))
        except Exception as e: rows.setdefault(n, {})[mode] = f'ERR {e}'
print('\nagents   routing-ON   routing-OFF   gain')
for n in sorted(rows):
    on=rows[n].get('on'); off=rows[n].get('off')
    g = (on/off-1)*100 if isinstance(on,(int,float)) and isinstance(off,(int,float)) and off else None
    print(f'{n:>6}   {on}   {off}   {g}')